#**2nd Week**

##**Задачи - CTE (Оба уровня)**

В этом тесте вам предстоит решить практические задачи на тему "CTE".

Найти клиентов, которые останавливались в номерах с мини-баром более двух раз. Вывести идентификатор клиента renter_id и колонку minibar_stays, в которой будет указано, сколько раз клиент останавливался в отеле в номерах с минибаром. Отсортировать по renter_id.

In [ ]:
SELECT 
    b.renter_id,
    COUNT(b.booking_id) AS minibar_stays
FROM 
    bookings b
JOIN 
    rooms r ON b.room_number = r.room_number
WHERE 
    r.has_minibar = 1
GROUP BY 
    b.renter_id
HAVING 
    COUNT(b.booking_id) > 2
ORDER BY 
    b.renter_id;

Вывести номера комнат, которые были оплачены методом "наличными" и имели общую сумму оплаты (amount_paid) больше средней суммы оплаты по всем номерам. Отсортируйте результат по возрастанию.

In [ ]:
WITH AveragePayment AS (
    SELECT AVG(price_per_night) AS avg_amount
    FROM rooms
),
RoomPayments AS (
    SELECT 
        b.room_number,
        SUM((julianday(b.check_out_date) - julianday(b.check_in_date)) * r.price_per_night) AS amount_paid
    FROM 
        bookings b
    JOIN 
        rooms r ON b.room_number = r.room_number
    WHERE 
        r.payment_option = 'на месте'  -- Payment method is cash
    GROUP BY 
        b.room_number
)
SELECT 
    rp.room_number
FROM 
    RoomPayments rp
JOIN 
    AveragePayment ap ON rp.amount_paid > ap.avg_amount
ORDER BY 
    rp.room_number;

Найти среднее количество бронирований в каждом месяце (на основе даты заезда). Вывести месяц (month) и колонку overall_avg_monthly_bookings, в которой будет указано среднее количество бронирований для данного месяца. Отсортировать в порядке убывания количества бронированийВыведите всю информацию о первых 50 бронированиях, для которых была совершена оплата в период с 1 января 2020 года по 31 марта 2020 года, отсортировав их по id бронирования.

In [ ]:
SELECT 
    strftime('%m', check_in_date) AS month,
    COUNT(booking_id) * 1.0 / COUNT(DISTINCT strftime('%Y', check_in_date)) AS overall_avg_monthly_bookings
FROM 
    bookings
GROUP BY 
    month
ORDER BY 
    overall_avg_monthly_bookings DESC;

Определить дни, когда было совершено больше всего платежей. Вывести дату платежа и количество платежей в этот день (payment_count). Отсортировать результат по дате платежа в порядке убывания.

In [ ]:
WITH daily_counts AS (
    SELECT 
        DATE(b.check_in_date) AS payment_date,
        COUNT(*) AS payment_count
    FROM 
        bookings b
    GROUP BY 
        DATE(b.check_in_date)
)
SELECT 
    payment_date, 
    payment_count
FROM 
    daily_counts
WHERE 
    payment_count = (SELECT MAX(payment_count) FROM daily_counts)
ORDER BY 
    payment_date DESC;


Найти клиентов с самым длинным именем и фамилией. Вывести идентификатор клиента, его имя и фамилию. Отсортировать по имени и фамилии.

Примечание: поиск следует производить по общей суммарной длине имени и фамилии.

In [ ]:
WITH ClientLengths AS (
    SELECT 
        id,
        first_name,
        last_name,
        LENGTH(first_name) + LENGTH(last_name) AS total_length
    FROM 
        clients
)
SELECT 
    id,
    first_name,
    last_name
FROM 
    ClientLengths
WHERE 
    total_length = (SELECT MAX(total_length) FROM ClientLengths)
ORDER BY 
    first_name, last_name;

Определить клиентов, которые не совершали бронирований, и добавить к ним флаг 'inactive'. Вывести идентификатор клиента, имя, фамилию и колонку client_status со статусом 'inactive'. Ограничьте вывод 50 записями.

In [ ]:
SELECT 
    c.id,
    c.first_name,
    c.last_name,
    'inactive' AS client_status
FROM 
    clients c
LEFT JOIN 
    bookings b ON c.id = b.renter_id
WHERE 
    b.booking_id IS NULL
LIMIT 50;

Найти минимальное количество прибыли для каждого месяца. Вывести месяц (month) и колонку min_monthly_amount_paid, в которой будет указано минимальная сумма платежей для данного месяца. Отсортировать в порядке возрастания суммы платежей.

In [ ]:
WITH monthly_profits AS (
    SELECT 
        strftime('%m', b.check_in_date) AS month,
        SUM((julianday(b.check_out_date) - julianday(b.check_in_date)) * r.price_per_night) AS total_profit
    FROM 
        bookings b
    JOIN 
        rooms r ON b.room_number = r.room_number
    GROUP BY 
        month
)
SELECT 
    month,
    MIN(total_profit) AS min_monthly_amount_paid
FROM 
    monthly_profits
GROUP BY 
    month
ORDER BY 
    min_monthly_amount_paid ASC;


Определить количество комнат, цена которых меньше средней цены по всем комнатам. Итоговое значение отобразить в колонке low_cost_rooms

In [ ]:
SELECT 
    COUNT(*) AS low_cost_rooms
FROM 
    rooms
WHERE 
    price_per_night < (SELECT AVG(price_per_night) FROM rooms);

Определить самые дорогие номера каждого типа. Вывести номер комнаты, ее тип и цену. Отсортировать по номеру комнаты.

In [ ]:
SELECT room_number, type_name , price_per_night 
FROM rooms
WHERE (type_name , price_per_night ) IN (
    SELECT type_name , MAX(price_per_night )
    FROM rooms
    GROUP BY type_name 
)
ORDER BY room_number;
